In [1]:
import sys, os
from dotenv import load_dotenv
load_dotenv()
print(sys.executable)
print("Key loaded:", os.getenv("GOOGLE_API_KEY")[:8] + "..." if os.getenv("GOOGLE_API_KEY") else "NOT FOUND")

C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe
Key loaded: AQ.Ab8RN...


In [2]:
import sys
!{sys.executable} -m pip install pypdf numpy langchain-text-splitters google-genai python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [3]:
from pypdf import PdfReader

def extract_text_by_page(pdf_path: str) -> list[dict]:
    
    """Returns a list of {page_number, text} dicts."""
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append({"page_number": i, "text": text.strip()})
    return pages

pages = extract_text_by_page("data/sample_docs/1706.03762v7.pdf")
print(f"Extracted {len(pages)} pages")
print(pages[0]["text"][:500])


Extracted 15 pages
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [4]:
for p in pages:
    print(f"Page {p['page_number']}: {len(p['text'])} characters")

Page 1: 2857 characters
Page 2: 4251 characters
Page 3: 1823 characters
Page 4: 2491 characters
Page 5: 3181 characters
Page 6: 3450 characters
Page 7: 3300 characters
Page 8: 3178 characters
Page 9: 2973 characters
Page 10: 3111 characters
Page 11: 3216 characters
Page 12: 3220 characters
Page 13: 812 characters
Page 14: 815 characters
Page 15: 818 characters


In [5]:

pip install langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_pages(pages: list[dict], chunk_size: int = 1000, chunk_overlap: int = 150) -> list[dict]:
    """Splits each page's text into overlapping chunks, keeping page_number as metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for page in pages:
        page_chunks = splitter.split_text(page["text"])
        for i, chunk_text in enumerate(page_chunks):
            chunks.append({
                "chunk_id": f"page{page['page_number']}_chunk{i}",
                "page_number": page["page_number"],
                "text": chunk_text,
            })
    return chunks

chunks = chunk_pages(pages)
print(f"Total chunks created: {len(chunks)}")
print(chunks[0])

Total chunks created: 49
{'chunk_id': 'page1_chunk0', 'page_number': 1, 'text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, th

In [7]:
print(chunks[5])
print("---")
print(chunks[20])

{'chunk_id': 'page2_chunk1', 'page_number': 2, 'text': 'sequence lengths, as memory constraints limit batching across examples. Recent work has achieved\nsignificant improvements in computational efficiency through factorization tricks [21] and conditional\ncomputation [32], while also improving model performance in case of the latter. The fundamental\nconstraint of sequential computation, however, remains.\nAttention mechanisms have become an integral part of compelling sequence modeling and transduc-\ntion models in various tasks, allowing modeling of dependencies without regard to their distance in\nthe input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms\nare used in conjunction with a recurrent network.\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead\nrelying entirely on an attention mechanism to draw global dependencies between input and output.\nThe Transformer allows for significantly 

In [8]:
smaller_chunks = chunk_pages(pages, chunk_size=500, chunk_overlap=100)
print(f"Total chunks with size=500: {len(smaller_chunks)}")
print(smaller_chunks[5])

Total chunks with size=500: 103
{'chunk_id': 'page1_chunk5', 'page_number': 1, 'text': 'has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head\nattention and the parameter-free position representation and became the other person involved in nearly every\ndetail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and\ntensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and'}


In [9]:
chunks = chunk_pages(pages, chunk_size=1000, chunk_overlap=150)
print(f"Using {len(chunks)} chunks for the rest of the pipeline")

Using 49 chunks for the rest of the pipeline


In [10]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: C:\Users\Vihu\docmind-ai
Files here: ['.env', '.git', '.gitignore', '.ipynb_checkpoints', '01_pdf_extraction.ipynb', 'app', 'data', 'docmind.db', 'notebooks', 'test_repo.py', 'venv']


In [11]:
import os
print(os.listdir("."))

['.env', '.git', '.gitignore', '.ipynb_checkpoints', '01_pdf_extraction.ipynb', 'app', 'data', 'docmind.db', 'notebooks', 'test_repo.py', 'venv']


In [12]:
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("GOOGLE_API_KEY")
print("Key loaded:", key[:8] + "..." if key else "NOT FOUND")

Key loaded: AQ.Ab8RN...


In [13]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents="This is a test sentence."
)
print("Embedding length:", len(result.embeddings[0].values))
print("First 5 values:", result.embeddings[0].values[:5])


Embedding length: 3072
First 5 values: [-0.038055073, -0.0028300586, 0.008414354, -0.0674523, -0.010631328]


In [14]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["The cat sat on the mat.", "A feline rested on the rug.", "Stock prices fell sharply today."]
)

emb1 = result.embeddings[0].values
emb2 = result.embeddings[1].values
emb3 = result.embeddings[2].values

print("Similar sentences similarity:", cosine_similarity(emb1, emb2))
print("Unrelated sentences similarity:", cosine_similarity(emb1, emb3))


Similar sentences similarity: 0.7705667422467881
Unrelated sentences similarity: 0.5598135856361813


In [15]:
import sys
print(sys.executable)


C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe


In [16]:
from google import genai
import os
import numpy as np
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["The cat sat on the mat.", "A feline rested on the rug.", "Stock prices fell sharply today."]
)

emb1 = result.embeddings[0].values
emb2 = result.embeddings[1].values
emb3 = result.embeddings[2].values

print("Similar sentences similarity:", cosine_similarity(emb1, emb2))
print("Unrelated sentences similarity:", cosine_similarity(emb1, emb3))

Similar sentences similarity: 0.7705667422467881
Unrelated sentences similarity: 0.5598135856361813


In [17]:
print("Pages:", len(pages))
print("Chunks:", len(chunks))


Pages: 15
Chunks: 49


In [18]:
import sys
!{sys.executable} -m pip install faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [19]:
def embed_chunks(chunks: list[dict], batch_size: int = 20) -> list[dict]:
    """Embeds each chunk's text, adding an 'embedding' key to each chunk dict."""
    texts = [c["text"] for c in chunks]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=batch
        )
        for j, emb in enumerate(result.embeddings):
            chunks[i + j]["embedding"] = emb.values
    return chunks

chunks = embed_chunks(chunks)
print(f"Embedded {len(chunks)} chunks")
print("Embedding dimension:", len(chunks[0]["embedding"]))

Embedded 49 chunks
Embedding dimension: 3072


In [20]:
import faiss
import numpy as np

def build_faiss_index(chunks: list[dict]):
    embeddings = np.array([c["embedding"] for c in chunks]).astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

index = build_faiss_index(chunks)
print(f"FAISS index built with {index.ntotal} vectors")

FAISS index built with 49 vectors


In [21]:
import sys
!{sys.executable} -m pip install faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Vihu\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [22]:
def search(query: str, k: int = 3):
    query_embedding = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query
    ).embeddings[0].values
    query_vector = np.array([query_embedding]).astype("float32")
    distances, indices = index.search(query_vector, k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "chunk": chunks[idx],
            "distance": dist
        })
    return results

results = search("What is multi-head attention?")
for r in results:
    print(f"Page {r['chunk']['page_number']} (distance: {r['distance']:.3f})")
    print(r['chunk']['text'][:200])
    print("---")

Page 5 (distance: 0.438)
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from diff
---
Page 4 (distance: 0.459)
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, 
---
Page 4 (distance: 0.505)
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Multi-Head Attention
Instead of performing a single attention function withdmodel-dimensional keys, va
---


In [23]:
import time, random
from google.genai import errors

MODELS = ["gemini-flash-lite-latest", "gemini-flash-latest", "gemini-3.8-flash"]

def build_prompt(query: str, retrieved: list[dict]) -> str:
    context = "\n\n".join(
        f"[Page {r['chunk']['page_number']}]\n{r['chunk']['text']}" for r in retrieved
    )
    return (
        "Answer the question using only the context below. "
        "Cite page numbers. If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )

def ask_document(query: str, k: int = 3, retries_per_model: int = 3) -> str:
    prompt = build_prompt(query, search(query, k=k))
    last_err = None

    for model in MODELS:
        for attempt in range(retries_per_model):
            try:
                response = client.models.generate_content(model=model, contents=prompt)
                return response.text
            except errors.APIError as e:
                last_err = e
                if e.code not in (429, 500, 503):   # real error (bad key, bad model name): don't retry
                    raise
                wait = 2 ** attempt * 2 + random.random()
                print(f"{model} attempt {attempt+1} failed ({e.code}), retrying in {wait:.1f}s...")
                time.sleep(wait)
        print(f"Giving up on {model}, trying next model...")

    raise last_err

print(ask_document("What is multi-head attention?"))


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the provided context, multi-head attention consists of several attention layers running in parallel (Page 4, Figure 2 caption). Instead of performing a single attention function with full-dimensionality keys, values, and queries, the model linearly projects the queries, keys, and values $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions respectively (Page 4). The attention function is then performed in parallel on each of these projected versions, yielding $d_v$-dimensional output values that are concatenated and projected again to result in the final values (Page 5). 

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions (Page 5).


In [24]:
answer2 = ask_document("What is the capital of France?")
print(answer2)

The answer isn't in the context.


In [25]:
def generate(prompt: str, retries_per_model: int = 3) -> str:
    """Model-fallback + retry logic, extracted from ask_document."""
    last_err = None
    for model in MODELS:
        for attempt in range(retries_per_model):
            try:
                return client.models.generate_content(model=model, contents=prompt).text
            except errors.APIError as e:
                last_err = e
                if e.code not in (429, 500, 503):
                    raise
                time.sleep(2 ** attempt * 2 + random.random())
    raise last_err

def condense_question(query: str, history: list[dict]) -> str:
    """Rewrite a follow-up into a standalone question using chat history."""
    if not history:
        return query
    convo = "\n".join(f"{m['role'].title()}: {m['content']}" for m in history[-6:])
    prompt = f"""Given the conversation and a follow-up question, rewrite the follow-up
as a single standalone question that makes sense without the conversation.
Return ONLY the rewritten question.

Conversation:
{convo}

Follow-up: {query}

Standalone question:"""
    return generate(prompt).strip()

history = []

def chat(query: str, k: int = 3) -> str:
    standalone = condense_question(query, history)
    print(f"[searching for: {standalone}]")
    answer = generate(build_prompt(standalone, search(standalone, k=k)))
    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": answer})
    return answer

print(chat("What is multi-head attention?"))
print(chat("How many of them does the base model use?"))

[searching for: What is multi-head attention?]
Based on the provided context, multi-head attention consists of several attention layers running in parallel (Page 4, Figure 2 caption). Instead of performing a single attention function with full-dimensionality keys, values, and queries, the model linearly projects the queries, keys, and values $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions, respectively (Pages 4–5). The attention function is then performed in parallel on each of these projected versions, and the resulting output values are concatenated and projected to produce the final values (Page 5). Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions (Page 5).
[searching for: How many attention heads does the base model use?]
Based on the provided context, the model employs $h = 8$ parallel attention layers, or heads (Page 5).


In [26]:
%%writefile app/db.py
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, sessionmaker

engine = create_engine("sqlite:///docmind.db")
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)

class Base(DeclarativeBase):
    pass

Overwriting app/db.py


In [27]:
%%writefile app/models.py
from datetime import datetime, timezone
from sqlalchemy import DateTime, ForeignKey, String, Text
from sqlalchemy.orm import Mapped, mapped_column, relationship
from app.db import Base

def utcnow() -> datetime:
    return datetime.now(timezone.utc)

class Conversation(Base):
    __tablename__ = "conversations"

    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200), default="New chat")
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), default=utcnow)
    messages: Mapped[list["Message"]] = relationship(
        back_populates="conversation",
        cascade="all, delete-orphan",
        order_by="Message.id",
    )

class Message(Base):
    __tablename__ = "messages"

    id: Mapped[int] = mapped_column(primary_key=True)
    conversation_id: Mapped[int] = mapped_column(ForeignKey("conversations.id"))
    role: Mapped[str] = mapped_column(String(20))      # "user" or "assistant"
    content: Mapped[str] = mapped_column(Text)
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), default=utcnow)
    conversation: Mapped["Conversation"] = relationship(back_populates="messages")

Overwriting app/models.py


In [28]:
%%writefile app/repositories.py
from sqlalchemy import select
from sqlalchemy.orm import Session
from app.models import Conversation, Message

class ConversationRepository:
    def __init__(self, session: Session):
        self.session = session

    def create(self, title: str = "New chat") -> Conversation:
        conv = Conversation(title=title)
        self.session.add(conv)
        self.session.commit()
        return conv

    def add_message(self, conversation_id: int, role: str, content: str) -> Message:
        msg = Message(conversation_id=conversation_id, role=role, content=content)
        self.session.add(msg)
        self.session.commit()
        return msg

    def get_history(self, conversation_id: int, limit: int = 6) -> list[dict]:
        """Last `limit` messages, oldest first, in the same shape the notebook uses."""
        stmt = (
            select(Message)
            .where(Message.conversation_id == conversation_id)
            .order_by(Message.id.desc())
            .limit(limit)
        )
        rows = list(self.session.scalars(stmt))[::-1]
        return [{"role": m.role, "content": m.content} for m in rows]

Overwriting app/repositories.py


In [29]:
%%writefile test_repo.py
from app.db import Base, SessionLocal, engine
import app.models  # registers the tables with Base
from app.repositories import ConversationRepository

Base.metadata.create_all(engine)

with SessionLocal() as session:
    repo = ConversationRepository(session)
    conv = repo.create("Attention paper")
    repo.add_message(conv.id, "user", "What is multi-head attention?")
    repo.add_message(conv.id, "assistant", "It runs several attention layers in parallel (Page 4).")
    repo.add_message(conv.id, "user", "How many of them does the base model use?")
    print("conversation id:", conv.id)
    print(repo.get_history(conv.id))

Overwriting test_repo.py


In [30]:
import os, sys
print("cwd:", os.getcwd())

try:
    import sqlalchemy
except ImportError:
    !{sys.executable} -m pip install sqlalchemy
    import sqlalchemy

from app.db import Base, SessionLocal, engine
import app.models
from app.repositories import ConversationRepository

Base.metadata.create_all(engine)

session = SessionLocal()
repo = ConversationRepository(session)
conv = repo.create("limit test")
repo.add_message(conv.id, "user", "What is multi-head attention?")
repo.add_message(conv.id, "assistant", "It runs several attention layers in parallel (Page 4).")
repo.add_message(conv.id, "user", "How many of them does the base model use?")

print(repo.get_history(conv.id, limit=2))

cwd: C:\Users\Vihu\docmind-ai
[{'role': 'assistant', 'content': 'It runs several attention layers in parallel (Page 4).'}, {'role': 'user', 'content': 'How many of them does the base model use?'}]


In [31]:
conv = repo.create("Attention paper chat")
print("conversation id:", conv.id)

def chat(query: str, conversation_id: int, k: int = 3) -> str:
    history = repo.get_history(conversation_id, limit=6)
    standalone = condense_question(query, history)
    print(f"[searching for: {standalone}]")
    answer = generate(build_prompt(standalone, search(standalone, k=k)))
    repo.add_message(conversation_id, "user", query)
    repo.add_message(conversation_id, "assistant", answer)
    return answer

print(chat("What is multi-head attention?", conv.id))
print(chat("How many of them does the base model use?", conv.id))

conversation id: 7
[searching for: What is multi-head attention?]
Based on the provided context, multi-head attention consists of several attention layers running in parallel (Page 4, Figure 2). Instead of performing a single attention function with full-dimensionality keys, values, and queries, it linearly projects the queries, keys, and values $h$ times with different, learned linear projections to $d_k$, $d_k$, and $d_v$ dimensions respectively (Page 4, Section 3.2.2). The attention function is then performed in parallel on each of these projected versions, and the resulting values are concatenated and projected to produce the final values (Page 5, Section 3.2.2). 

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions (Page 5).
[searching for: How many attention heads does the base model use in multi-head attention?]
Based on the provided context, the work employs $h = 8$ parallel attention layers, or h

In [32]:
import faiss, json

faiss.write_index(index, "data/index.faiss")
slim = [{k: v for k, v in c.items() if k != "embedding"} for c in chunks]
with open("data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(slim, f, ensure_ascii=False)
print("saved", index.ntotal, "vectors and", len(slim), "chunks")

saved 49 vectors and 49 chunks


In [33]:
%%writefile app/rag.py
import json, os, random, time
import faiss
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import errors

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODELS = ["gemini-flash-lite-latest", "gemini-flash-latest", "gemini-3.8-flash"]

index = faiss.read_index("data/index.faiss")
with open("data/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)


def search(query: str, k: int = 3) -> list[dict]:
    emb = client.models.embed_content(model="gemini-embedding-001", contents=query)
    vec = np.array([emb.embeddings[0].values]).astype("float32")
    distances, indices = index.search(vec, k)
    return [{"chunk": chunks[i], "distance": float(d)} for i, d in zip(indices[0], distances[0])]


def build_prompt(query: str, retrieved: list[dict]) -> str:
    context = "\n\n---\n\n".join(
        f"[Page {r['chunk']['page_number']}]\n{r['chunk']['text']}" for r in retrieved
    )
    return f"""You are a helpful assistant answering questions about a document.
Use ONLY the excerpts below to answer the question. Each excerpt is labeled with its page number.
If the answer is not contained in these excerpts, say "I couldn't find this in the document" - do not use outside knowledge.
When you answer, cite the page number(s) your answer comes from, like (Page 4).

Excerpts:
{context}

Question: {query}

Answer:"""


def generate(prompt: str, retries_per_model: int = 3) -> str:
    last_err = None
    for model in MODELS:
        for attempt in range(retries_per_model):
            try:
                return client.models.generate_content(model=model, contents=prompt).text
            except errors.APIError as e:
                last_err = e
                if e.code not in (429, 500, 503):
                    raise
                time.sleep(2 ** attempt * 2 + random.random())
    raise last_err


def condense_question(query: str, history: list[dict]) -> str:
    if not history:
        return query
    convo = "\n".join(f"{m['role'].title()}: {m['content']}" for m in history[-6:])
    prompt = f"""Given the conversation and a follow-up question, rewrite the follow-up
as a single standalone question that makes sense without the conversation.
Return ONLY the rewritten question.

Conversation:
{convo}

Follow-up: {query}

Standalone question:"""
    return generate(prompt).strip()


def ask(query: str, history: list[dict], k: int = 3) -> dict:
    standalone = condense_question(query, history)
    retrieved = search(standalone, k=k)
    answer = generate(build_prompt(standalone, retrieved))
    return {
        "answer": answer,
        "standalone_question": standalone,
        "pages": sorted({r["chunk"]["page_number"] for r in retrieved}),
    }

Overwriting app/rag.py


In [34]:
%%writefile app/main.py
from fastapi import Depends, FastAPI, HTTPException
from pydantic import BaseModel
from sqlalchemy.orm import Session

from app import rag
from app.db import Base, SessionLocal, engine
from app.models import Conversation
from app.repositories import ConversationRepository

Base.metadata.create_all(engine)
app = FastAPI(title="DocMind AI")


def get_repo():
    with SessionLocal() as session:
        yield ConversationRepository(session)


class ChatRequest(BaseModel):
    question: str


@app.post("/conversations")
def create_conversation(repo: ConversationRepository = Depends(get_repo)):
    conv = repo.create("New chat")
    return {"id": conv.id}


@app.post("/conversations/{conversation_id}/chat")
def chat(conversation_id: int, body: ChatRequest, repo: ConversationRepository = Depends(get_repo)):
    if repo.session.get(Conversation, conversation_id) is None:
        raise HTTPException(status_code=404, detail="Conversation not found")
    history = repo.get_history(conversation_id, limit=6)
    result = rag.ask(body.question, history)
    repo.add_message(conversation_id, "user", body.question)
    repo.add_message(conversation_id, "assistant", result["answer"])
    return result

Overwriting app/main.py


In [36]:
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, sessionmaker

engine = create_engine("sqlite:///docmind.db")
SessionLocal = sessionmaker(bind=engine, expire_on_commit=False)

class Base(DeclarativeBase):
    pass
    

In [39]:
with open("app/repositories.py") as f:
    print(f.read())

from sqlalchemy import select
from sqlalchemy.orm import Session
from app.models import Conversation, Message

class ConversationRepository:
    def __init__(self, session: Session):
        self.session = session

    def create(self, title: str = "New chat") -> Conversation:
        conv = Conversation(title=title)
        self.session.add(conv)
        self.session.commit()
        return conv

    def add_message(self, conversation_id: int, role: str, content: str) -> Message:
        msg = Message(conversation_id=conversation_id, role=role, content=content)
        self.session.add(msg)
        self.session.commit()
        return msg

    def get_history(self, conversation_id: int, limit: int = 6) -> list[dict]:
        """Last `limit` messages, oldest first, in the same shape the notebook uses."""
        stmt = (
            select(Message)
            .where(Message.conversation_id == conversation_id)
            .order_by(Message.id.desc())
            .limit(limit)
        )